In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten,BatchNormalization,Dropout
from sklearn.model_selection import train_test_split
import random
seed_value = 42
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

"""import random
tf.random.set_seed(42) 
import numpy as np 
np.random.seed(42) """

# Parameters
image_size = (64, 64)
batch_size = 32
test_ratio = 0.1

# Directories
data_dir = r"C:\Users\hrish\Downloads\homer_bart"
class_names = os.listdir(data_dir)

# Function to load and preprocess images
def load_and_preprocess_image(img_path):
    img = tf.keras.preprocessing.image.load_img(img_path,color_mode='grayscale', target_size=image_size)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = img_array / 255.0  # Normalize the image to [0, 1]

    return img_array

# Load images and labels
images = []
labels = []
for class_index, class_name in enumerate(class_names):
    class_dir = os.path.join(data_dir, class_name)
    for img_name in os.listdir(class_dir):
        img_path = os.path.join(class_dir, img_name)
        img_array = load_and_preprocess_image(img_path)
        images.append(img_array)
        labels.append(class_index)

images = np.array(images)
labels = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=test_ratio, random_state=42)

datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Fit the data generator on training data
datagen.fit(X_train)

# Create the model
model = Sequential([
    Flatten(input_shape=(64, 64, 1)),  # 1 channel for grayscale
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')

])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
# Compile the model
model.compile(optimizer=optimizer,
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=50, batch_size=batch_size, validation_data=(X_test, y_test))

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {test_accuracy}')